# 🛠️ Geração de Dataset Sintético para Treinamento de Machine Learning

Para cumprir os requisitos do desafio relacionados ao treinamento de modelos preditivos e à utilização de dados anonimizados/sintéticos, o bloco de código abaixo implementa um motor de geração de dados estruturados.

Como o **Guardiã AI** utiliza variáveis acústicas inéditas (como a taxa de hesitação vocal) que não estão presentes em bancos de dados de saúde tradicionais, optou-se pela criação de um dataset 100% sintético e customizado que simula o fluxo de pacientes em uma triagem médica.

### ⚙️ Estrutura da Geração de Dados:
1. **Dados Demográficos e de Identificação:** Utilização da biblioteca `Faker` para criar perfis realistas, porém totalmente fictícios e anonimizados (Nomes, RG, CPF, Idade, UF).
2. **Sinais Vitais (Base Biológica):** Simulação de métricas clínicas (Pressão Arterial, Frequência Cardíaca, SpO2, etc.) através da aplicação de distribuições normais estatísticas (`numpy.random.normal`), garantindo que os valores fiquem dentro de limites fisiológicos aceitáveis.
3. **Features Acústicas Inéditas:** Injeção de variáveis de 0 a 100% simulando as métricas extraídas no fluxo do áudio (ex: `taxa_hesitacao_porcento`).
4. **Criação da Variável Alvo (`urgencia_critica`):** Aplicação de uma regra de negócio clínica determinística para rotular o nível de risco. Pacientes com sinais vitais deteriorados ou com alta hesitação associada à taquicardia recebem o *label* de risco alto (`1`).
5. **Correlação Lógica:** Geração de diagnósticos e histórico de internação enviesados pela variável alvo, facilitando o aprendizado de padrões pelo modelo de Machine Learning.
6. **Exportação:** Salvamento automático no formato `.csv` com disparo de download direto para a máquina local através das ferramentas do Google Colab.

# 📦 Instalação de Dependências: Biblioteca Faker

Para gerar os dados de identificação do nosso dataset sintético de forma realista e segura (criando nomes, CPFs, RGs e datas de nascimento totalmente fictícios), utilizaremos a biblioteca `Faker`.

Como esta ferramenta não vem pré-instalada por padrão no ambiente do Google Colab, precisamos realizar a sua instalação via gerenciador de pacotes (`pip`) antes de prosseguir com o motor de geração de dados.

In [3]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.7 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime
import random
import re
from google.colab import files

# 1. Configuração Inicial
fake = Faker('pt_BR')
Faker.seed(42)
np.random.seed(42)

# Definir a quantidade de registros do dataset
num_registros = 5000

# Função para remover abreviaturas e pronomes de tratamento gerados pelo Faker
def gerar_nome_limpo():
    nome = fake.name()
    # Remove prefixos comuns no início da string
    nome_limpo = re.sub(r'^(Sr\.|Sra\.|Srta\.|Dr\.|Dra\.|Prof\.|Profa\.)\s+', '', nome)
    return nome_limpo

# 2. Geração de Dados Gerais e Demográficos
dados = {
    'nome_completo': [gerar_nome_limpo() for _ in range(num_registros)],
    'data_nascimento': [fake.date_of_birth(minimum_age=18, maximum_age=90) for _ in range(num_registros)],
    'rg': [fake.rg() for _ in range(num_registros)],
    'cpf': [fake.cpf() for _ in range(num_registros)],
    'data_atendimento': [fake.date_time_between(start_date='-1y', end_date='now').strftime('%Y-%m-%d %H:%M:%S') for _ in range(num_registros)],
    'uf_atendimento': [fake.estado_sigla() for _ in range(num_registros)],
    'modo_chegada': [random.choice([
        'Chegou ao pronto-socorro por meios próprios.',
        'Trazida pelo resgate (SAMU).',
        'Encaminhamento via viatura policial.',
        'Acompanhada por familiares.'
    ]) for _ in range(num_registros)],
    'historico_comorbidades': [random.choice(['Nenhuma', 'Hipertensão', 'Diabetes', 'Doença Cardiovascular', 'Asma']) for _ in range(num_registros)]
}

df = pd.DataFrame(dados)

# Engenharia de features: calcular Idade
df['idade'] = df['data_nascimento'].apply(lambda x: (datetime.now().date() - x).days // 365)

# 3. Geração de Sinais Vitais
df['pressao_sistolica'] = np.clip(np.random.normal(loc=120, scale=20, size=num_registros), 70, 220).astype(int)
df['pressao_diastolica'] = np.clip(np.random.normal(loc=80, scale=15, size=num_registros), 40, 130).astype(int)
df['frequencia_cardiaca'] = np.clip(np.random.normal(loc=85, scale=25, size=num_registros), 40, 180).astype(int)
df['frequencia_respiratoria'] = np.clip(np.random.normal(loc=18, scale=5, size=num_registros), 10, 40).astype(int)
df['temperatura_celsius'] = np.round(np.clip(np.random.normal(loc=36.5, scale=0.8, size=num_registros), 34.0, 41.0), 1)
df['spo2_porcento'] = np.clip(np.random.normal(loc=97, scale=4, size=num_registros), 70, 100).astype(int)

# 4. Geração de Features Acústicas
df['taxa_hesitacao_porcento'] = np.round(np.random.uniform(0, 100, size=num_registros), 2)
df['instabilidade_emocional_vocal_porcento'] = np.round(np.random.uniform(0, 100, size=num_registros), 2)

# 5. Aplicação da Regra de Negócio Clínica (Variável Alvo)
condicao_risco = (df['spo2_porcento'] < 93) | ((df['taxa_hesitacao_porcento'] > 40) & (df['frequencia_cardiaca'] > 120))
df['urgencia_critica'] = np.where(condicao_risco, 1, 0)

# 6. Geração de Variáveis Dependentes (Pós-Triagem)
diagnosticos_criticos = ['Infarto Agudo do Miocárdio', 'Insuficiência Respiratória', 'Crise Hipertensiva Severa', 'Trauma Físico Severo']
diagnosticos_seguros = ['Crise de Ansiedade', 'Mialgia (Dor Muscular)', 'Refluxo Gastroesofágico', 'Cefaleia Tensional']

df['diagnostico_realizado'] = df['urgencia_critica'].apply(
    lambda x: random.choice(diagnosticos_criticos) if x == 1 else random.choice(diagnosticos_seguros)
)

df['houve_internacao'] = df['urgencia_critica'].apply(
    lambda x: np.random.choice([1, 0], p=[0.92, 0.08]) if x == 1 else np.random.choice([1, 0], p=[0.05, 0.95])
)

# 7. Organização e Exportação
colunas_ordenadas = [
    'nome_completo', 'data_nascimento', 'idade', 'rg', 'cpf', 'uf_atendimento',
    'data_atendimento', 'modo_chegada', 'historico_comorbidades',
    'pressao_sistolica', 'pressao_diastolica', 'frequencia_cardiaca',
    'frequencia_respiratoria', 'temperatura_celsius', 'spo2_porcento',
    'taxa_hesitacao_porcento', 'instabilidade_emocional_vocal_porcento',
    'diagnostico_realizado', 'houve_internacao', 'urgencia_critica'
]
df = df[colunas_ordenadas]

# Exportar para arquivo CSV
nome_arquivo = 'dataset_guardia_ai_triagem.csv'
df.to_csv(nome_arquivo, index=False, encoding='utf-8')

print(f"✅ Dataset sintético gerado com sucesso: {nome_arquivo}")
print(f"📊 Distribuição da Variável Alvo (Urgência Crítica):")
print(df['urgencia_critica'].value_counts(normalize=True) * 100)

# 8. Download Automático (Específico para Google Colab)
files.download(nome_arquivo)

✅ Dataset sintético gerado com sucesso: dataset_guardia_ai_triagem.csv
📊 Distribuição da Variável Alvo (Urgência Crítica):
urgencia_critica
0    79.58
1    20.42
Name: proportion, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>